In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier

In [2]:

from google.colab import files
import pandas as pd

uploaded = files.upload()

df = pd.read_csv('diabetes.csv')

print(df.head())
print(df.shape)

Saving diabetes.csv to diabetes.csv
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)


In [3]:

# LOAD DATA

# ==========================================


df = pd.read_csv("diabetes.csv")

print(df.head())
print(df.shape)

#

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)


In [4]:

# FEATURES & TARGET
# ==========================================

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [5]:

# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [6]:

# ==========================================
# PIPELINE
# ==========================================

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=42
    ))
])

#

In [8]:

# HYPERPARAMETER TUNING
# ==========================================

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_

Best Parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 100}


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:

# EVALUATION
# ==========================================

y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)[:,1]

print("\nAccuracy:",
      accuracy_score(y_test, y_pred))

print("\nROC-AUC:",
      roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.7597402597402597

ROC-AUC: 0.8305555555555556

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.84      0.82       100
           1       0.67      0.61      0.64        54

    accuracy                           0.76       154
   macro avg       0.74      0.73      0.73       154
weighted avg       0.76      0.76      0.76       154


Confusion Matrix:
[[84 16]
 [21 33]]


In [16]:

# SAVE MODEL
# ==========================================

joblib.dump(
    best_model,
    "diabetes_model.pkl"
)

print("\nModel Saved!")


Model Saved!


In [14]:
import joblib
import pandas as pd

model = joblib.load("diabetes_model.pkl")

def predict_diabetes(
    pregnancies,
    glucose,
    blood_pressure,
    skin_thickness,
    insulin,
    bmi,
    diabetes_pedigree,
    age
):

    data = pd.DataFrame({
        "Pregnancies":[pregnancies],
        "Glucose":[glucose],
        "BloodPressure":[blood_pressure],
        "SkinThickness":[skin_thickness],
        "Insulin":[insulin],
        "BMI":[bmi],
        "DiabetesPedigreeFunction":[diabetes_pedigree],
        "Age":[age]
    })

    probability = model.predict_proba(data)[0][1]

    prediction = model.predict(data)[0]

    return {
        "Prediction":
            "Diabetic" if prediction == 1 else "Non-Diabetic",

        "Probability":
            round(probability*100,2)
    }

In [15]:
result = predict_diabetes(
    pregnancies=2,
    glucose=120,
    blood_pressure=70,
    skin_thickness=20,
    insulin=85,
    bmi=28.5,
    diabetes_pedigree=0.35,
    age=35
)

print(result)

{'Prediction': 'Non-Diabetic', 'Probability': np.float32(25.96)}
